# CNN_5 Baseline (ไม่ทำ Gaussian) : Sleep Stage Classification (HMC)

## 1. Import

In [ ]:
!pip install -q mne

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.signal import resample_poly
import mne
from matplotlib.lines import Line2D
from matplotlib.ticker import MultipleLocator

import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_fscore_support,
)

## 2. ตั้งค่า

In [ ]:
PROJECT_DIR = Path("/content/drive/MyDrive/SleepStageClassificationProject")
DATA_DIR = PROJECT_DIR / "dataset" / "preprocess_microvolt"
RESULT_DIR = PROJECT_DIR / "results" / "CNN_5_4ch_seed43"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 43
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

FS_RAW = 256
FS = 100
EPOCH_LEN_RAW = 30 * FS_RAW
EPOCH_LEN = 30 * FS
SEQ_LEN = 5
SEQ_SAMPLES = SEQ_LEN * EPOCH_LEN

ALL_CHANNELS = ["F4-M1", "C4-M1", "O2-M1", "C3-M2", "E1-M2", "E2-M2", "Chin"]
SELECTED_CHANNELS = ["C4-M1", "C3-M2", "E1-M2", "Chin"]
CHANNEL_IDX = [ALL_CHANNELS.index(c) for c in SELECTED_CHANNELS]
N_ALL_CHANNELS = len(ALL_CHANNELS)
N_CHANNELS = len(SELECTED_CHANNELS)

STAGES = ["W", "N1", "N2", "N3", "R"]
LABELS = [0, 1, 2, 3, 4]
SPLITS = ["train", "val", "test"]
EXPECTED_SUBJECTS = {"train": 79, "val": 21, "test": 23}

CNN_FILTERS = [8, 16, 32]
KERNEL_SIZE = 100
DENSE_UNITS = 50
DROPOUT = 0.5

BATCH_SIZE = 80
MAX_TRAIN_EPOCHS = 30
EVALS_PER_EPOCH = 5
LEARNING_RATE = 0.001
LR_DECAY_EVERY = 10
MIN_LR = 1e-6
PATIENCE = 10

## 3. โหลดข้อมูล (เฉพาะ subject ที่มีครบ 5 ระยะ ระยะละ 5 epoch ติดกัน)

In [ ]:
def load_split(split):
    X_list = []
    rows = []

    for path in sorted((DATA_DIR / split).glob("*.npz")):
        subject = path.stem
        subject_X = []
        subject_rows = []

        with np.load(path, allow_pickle=True) as data:
            stages = data["stage"]
            n_valid = data["n_valid"]
            y_all = data["y"]
            epochs_all = data["epoch_indices"]
            x_all = data["X"]

            for g in range(len(stages)):
                stage = str(stages[g])

                if stage not in STAGES:
                    continue
                if int(n_valid[g]) != SEQ_LEN:
                    continue

                y_group = np.asarray(y_all[g]).astype(str).ravel()
                epochs = np.asarray(epochs_all[g]).astype(int).ravel()

                if len(y_group) != SEQ_LEN or not np.all(y_group == stage):
                    continue
                if len(epochs) != SEQ_LEN or not np.all(np.diff(epochs) == 1):
                    continue

                x = np.asarray(x_all[g], dtype=np.float32)

                if x.shape != (SEQ_LEN, EPOCH_LEN_RAW, N_ALL_CHANNELS):
                    continue
                if not np.isfinite(x).all():
                    continue

                x = x[:, :, CHANNEL_IDX]
                x = x.reshape(SEQ_LEN * EPOCH_LEN_RAW, N_CHANNELS)
                x = resample_poly(x, FS, FS_RAW, axis=0)

                subject_X.append(x.astype(np.float32))
                subject_rows.append({
                    "subject": subject,
                    "stage": stage,
                    "label": STAGES.index(stage),
                    "epoch_start": int(epochs[0]),
                    "epoch_end": int(epochs[-1]),
                })

        found_stages = set(row["stage"] for row in subject_rows)
        if found_stages == set(STAGES):
            X_list.extend(subject_X)
            rows.extend(subject_rows)

    return np.stack(X_list), pd.DataFrame(rows)


DATA = {}
for split in SPLITS:
    X, meta = load_split(split)
    DATA[split] = {"X": X, "y": meta["label"].to_numpy().astype(np.int32), "meta": meta}
    print(split, "| subjects:", meta["subject"].nunique(), "| sequences:", len(meta), "| X shape:", X.shape)

actual_subjects = {split: DATA[split]["meta"]["subject"].nunique() for split in SPLITS}
assert actual_subjects == EXPECTED_SUBJECTS, actual_subjects

for split in SPLITS:
    meta = DATA[split]["meta"]
    assert meta.groupby(["subject", "stage"]).size().eq(1).all(), split
    assert meta.groupby("subject").size().eq(len(STAGES)).all(), split

for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    assert not set(DATA[a]["meta"]["subject"]) & set(DATA[b]["meta"]["subject"])

X_train, y_train = DATA["train"]["X"], DATA["train"]["y"]
X_val, y_val = DATA["val"]["X"], DATA["val"]["y"]
X_test, y_test = DATA["test"]["X"], DATA["test"]["y"]

display(pd.DataFrame({
    "channel": SELECTED_CHANNELS,
    "min": X_train.min(axis=(0, 1)),
    "max": X_train.max(axis=(0, 1)),
    "mean": X_train.mean(axis=(0, 1)),
    "sd": X_train.std(axis=(0, 1)),
}))

## 4. สร้างโมเดล CNN_5

In [ ]:
def build_cnn5():
    inputs = layers.Input(shape=(SEQ_SAMPLES, N_CHANNELS))
    x = layers.Permute((2, 1))(inputs)
    x = layers.Reshape((N_CHANNELS, SEQ_SAMPLES, 1))(x)

    for n_filters in CNN_FILTERS:
        x = layers.Conv2D(n_filters, (1, KERNEL_SIZE), strides=1, padding="same")(x)
        x = layers.ReLU()(x)
        x = layers.BatchNormalization()(x)
        x = layers.AveragePooling2D(pool_size=(1, 2), strides=(1, 2))(x)

    x = layers.Flatten()(x)
    x = layers.Dense(DENSE_UNITS)(x)
    x = layers.ReLU()(x)
    x = layers.Dropout(DROPOUT)(x)
    outputs = layers.Dense(len(STAGES), activation="softmax")(x)

    return models.Model(inputs, outputs, name="CNN_5")


model = build_cnn5()
model.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=LEARNING_RATE),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

## 5. ฝึกโมเดล (เลือก checkpoint จาก validation loss)

In [ ]:
n_train = len(y_train)
batches_per_epoch = int(np.ceil(n_train / BATCH_SIZE))
eval_every = max(1, round(batches_per_epoch / EVALS_PER_EPOCH))

evals_per_epoch = batches_per_epoch / eval_every

print("batches per training epoch:", batches_per_epoch)
print("validate every", eval_every, "batch(es)")
print("validations per training epoch:", evals_per_epoch)
print("patience:", PATIENCE, "validations =", PATIENCE / evals_per_epoch, "training epochs")

history = []
best_val_loss = np.inf
best_weights = model.get_weights()
wait = 0
step = 0
stop = False

for epoch in range(MAX_TRAIN_EPOCHS):
    lr = max(LEARNING_RATE * 0.1 ** (epoch // LR_DECAY_EVERY), MIN_LR)
    model.optimizer.learning_rate.assign(lr)

    order = np.random.permutation(n_train)

    for b in range(batches_per_epoch):
        idx = order[b * BATCH_SIZE:(b + 1) * BATCH_SIZE]
        result = model.train_on_batch(X_train[idx], y_train[idx], return_dict=True)
        step += 1

        if step % eval_every == 0:
            val_loss, val_acc = model.evaluate(X_val, y_val, batch_size=BATCH_SIZE, verbose=0)
            history.append({
                "step": step,
                "train_epoch": step / batches_per_epoch,
                "lr": lr,
                "train_last_batch_loss": result["loss"],
                "train_last_batch_accuracy": result["accuracy"],
                "val_loss": val_loss,
                "val_accuracy": val_acc,
            })

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_weights = model.get_weights()
                wait = 0
            else:
                wait += 1

            if wait >= PATIENCE:
                stop = True
                break

    print(f"epoch {epoch + 1}: val_loss={history[-1]['val_loss']:.4f}, val_acc={history[-1]['val_accuracy']:.4f}, best_val_loss={best_val_loss:.4f}")

    if stop:
        print("early stopping")
        break

model.set_weights(best_weights)
model.save(RESULT_DIR / "cnn5_best_model.keras")

history_df = pd.DataFrame(history)
history_df.to_csv(RESULT_DIR / "training_history.csv", index=False)

best_row = history_df.loc[history_df["val_loss"].idxmin()]
best_step = best_row["step"]
print("best checkpoint at step", int(best_step),
      "| val_loss", round(best_val_loss, 4),
      "| val_accuracy", round(best_row["val_accuracy"], 4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, name in zip(axes, ["loss", "accuracy"]):
    ax.plot(history_df["step"], history_df[f"train_last_batch_{name}"], label="train (last batch)")
    ax.plot(history_df["step"], history_df[f"val_{name}"], label="validation")
    ax.axvline(best_step, color="gray", linestyle="--", label="best checkpoint")
    ax.set_xlabel("Training update step")
    ax.set_ylabel(name)
    ax.legend()
    ax.grid(True)
plt.tight_layout()
plt.savefig(RESULT_DIR / "training_curves.png", dpi=150)
plt.show()

## 6. ฟังก์ชันคำนวณ metric

In [ ]:
def calc_metrics(y_true, y_pred):
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=LABELS, zero_division=0
    )
    overall = {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": precision.mean(),
        "macro_recall": recall.mean(),
        "macro_f1": f1.mean(),
        "cohen_kappa": cohen_kappa_score(y_true, y_pred, labels=LABELS),
    }
    per_class = pd.DataFrame({
        "stage": STAGES,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "support": support,
    })
    return overall, per_class


def stage_confusion(cm):
    rows = []
    total = cm.sum()
    for i, stage in enumerate(STAGES):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        tn = total - tp - fp - fn
        rows.append({
            "stage": stage,
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "TN": tn,
            "sensitivity": tp / (tp + fn) if tp + fn > 0 else 0,
            "specificity": tn / (tn + fp) if tn + fp > 0 else 0,
        })
    return pd.DataFrame(rows)


def class_analysis(cm):
    rows = []
    for i, stage in enumerate(STAGES):
        errors = cm[i].copy()
        errors[i] = 0
        j = errors.argmax()
        rows.append({
            "stage": stage,
            "correct": int(cm[i, i]),
            "total": int(cm[i].sum()),
            "recall": cm[i, i] / cm[i].sum(),
            "most_confused_with": STAGES[j] if errors[j] > 0 else "-",
            "confused_count": int(errors[j]),
        })
    return pd.DataFrame(rows)


def make_prediction_table(split, prob):
    meta = DATA[split]["meta"]
    pred = prob.argmax(axis=1)
    table = meta[["subject", "epoch_start", "epoch_end"]].copy()
    table["actual"] = [STAGES[i] for i in meta["label"]]
    table["predicted"] = [STAGES[i] for i in pred]
    table["confidence"] = prob.max(axis=1)
    for i, stage in enumerate(STAGES):
        table["prob_" + stage] = prob[:, i]
    table["correct"] = table["actual"] == table["predicted"]
    return table

## 7. ประเมินผล Train, Validation และ Test

In [ ]:
model = tf.keras.models.load_model(RESULT_DIR / "cnn5_best_model.keras")

results = {}
per_class_all = {}
tables = {}

for split in SPLITS:
    X, y = DATA[split]["X"], DATA[split]["y"]
    CHUNK = 1000
    prob = np.concatenate([
        model.predict(X[i:i + CHUNK], batch_size=BATCH_SIZE, verbose=0)
        for i in range(0, len(X), CHUNK)
    ])
    y_pred = prob.argmax(axis=1)

    overall, per_class = calc_metrics(y, y_pred)
    loss = -np.log(np.clip(prob[np.arange(len(y)), y], 1e-7, 1)).mean()

    results[split] = {"loss": loss, **overall}
    per_class_all[split] = per_class
    tables[split] = make_prediction_table(split, prob)

    pd.DataFrame([results[split]]).to_csv(RESULT_DIR / f"{split}_overall_metrics.csv", index=False)

    if split == "train":
        continue

    cm = confusion_matrix(y, y_pred, labels=LABELS)
    cm_norm = confusion_matrix(y, y_pred, labels=LABELS, normalize="true")

    print("=====", split.upper(), "=====")
    display(pd.DataFrame([results[split]]).round(4))

    print("Per-class metrics")
    display(per_class.round(4))

    print("Per-stage confusion (one-vs-rest)")
    display(stage_confusion(cm).round(4))

    print("Per-class analysis")
    display(class_analysis(cm).round(4))

    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
    ConfusionMatrixDisplay(cm, display_labels=STAGES).plot(ax=axes[0], cmap="Blues", colorbar=False)
    axes[0].set_title("Confusion matrix (count)")
    ConfusionMatrixDisplay(cm_norm, display_labels=STAGES).plot(ax=axes[1], cmap="Blues", values_format=".2f", colorbar=False)
    axes[1].set_title("Confusion matrix (normalized by actual)")
    plt.suptitle(split)
    plt.tight_layout()
    plt.savefig(RESULT_DIR / f"{split}_confusion_matrix.png", dpi=150)
    plt.show()

    x_pos = np.arange(len(STAGES))
    plt.figure(figsize=(8, 4.5))
    plt.bar(x_pos - 0.27, per_class["precision"], 0.27, label="Precision")
    plt.bar(x_pos, per_class["recall"], 0.27, label="Recall")
    plt.bar(x_pos + 0.27, per_class["f1_score"], 0.27, label="F1-score")
    plt.xticks(x_pos, STAGES)
    plt.ylim(0, 1.05)
    plt.title(f"{split}: metrics per stage")
    plt.legend()
    plt.grid(axis="y", alpha=0.4)
    plt.tight_layout()
    plt.savefig(RESULT_DIR / f"{split}_per_class_metrics.png", dpi=150)
    plt.show()

    per_class.to_csv(RESULT_DIR / f"{split}_per_class_metrics.csv", index=False)
    stage_confusion(cm).to_csv(RESULT_DIR / f"{split}_per_stage_confusion.csv", index=False)
    class_analysis(cm).to_csv(RESULT_DIR / f"{split}_class_analysis.csv", index=False)
    pd.DataFrame(cm, index=STAGES, columns=STAGES).to_csv(RESULT_DIR / f"{split}_confusion_matrix.csv")
    tables[split].to_csv(RESULT_DIR / f"{split}_predictions.csv", index=False)


## 7.1 เปรียบเทียบ Train / Validation / Test

In [ ]:
summary = pd.DataFrame(results).T
summary.index.name = "split"
summary.to_csv(RESULT_DIR / "baseline_summary.csv")

print("Overall metrics")
display(summary.round(4))

f1_compare = pd.DataFrame({split: per_class_all[split].set_index("stage")["f1_score"] for split in SPLITS})
f1_compare.to_csv(RESULT_DIR / "f1_per_stage_train_val_test.csv")

print("F1-score per stage")
display(f1_compare.round(4))


## 7.2 แถวสรุปผลสำหรับตารางบันทึกการทดลอง (Train, Validation และ Test)

In [ ]:
row = {
    "experiment": "CNN_5_4ch_seed43",
    "SEED": SEED,
    "changed_factor": "baseline",
    "n_train_sequences": len(y_train),
    "trainable_params": int(sum(np.prod(w.shape) for w in model.trainable_weights)),
    "best_val_loss": round(best_val_loss, 4),
}

for split in SPLITS:
    row[f"{split}_accuracy"] = round(results[split]["accuracy"], 4)
    row[f"{split}_macro_f1"] = round(results[split]["macro_f1"], 4)
    row[f"{split}_kappa"] = round(results[split]["cohen_kappa"], 4)

summary_row = pd.DataFrame([row])

display(summary_row)
summary_row.to_csv(RESULT_DIR / "experiment_row.csv", index=False)


## 8. ตารางผลทำนายราย sequence (Test)

In [ ]:
test_table = tables["test"]
print("correct:", int(test_table["correct"].sum()), "/", len(test_table))
display(test_table.round(3))

## 9. Hypnogram เทียบระยะจริงกับที่โมเดลทำนาย (Test)

In [ ]:
stage_y = {"W": 4, "R": 3, "N1": 2, "N2": 1, "N3": 0}

hyp_dir = RESULT_DIR / "hypnogram"
hyp_dir.mkdir(parents=True, exist_ok=True)

FULL_NIGHT_DIR = PROJECT_DIR / "dataset" / "Test_Full_night"
EPOCH_SEC = 30

FIG_W, FIG_H = 30, 10
GT_COLOR = "#1f5fff"
PRED_LINE_COLOR = "black"
LINE_W = 1.5
BLOCK_W = 8
MARKER_SIZE = 90
PRED_MARKER_SIZE = 40
DASH = (0, (5, 3))

ann_to_stage = {
    "sleep stage w": "W",
    "sleep stage n1": "N1", "sleep stage 1": "N1",
    "sleep stage n2": "N2", "sleep stage 2": "N2",
    "sleep stage n3": "N3", "sleep stage 3": "N3",
    "sleep stage r": "R",
}


def load_full_night(subject):
    ann = mne.read_annotations(str(FULL_NIGHT_DIR / f"{subject}_sleepscoring.edf"))

    items = []
    for onset, dur, desc in zip(ann.onset, ann.duration, ann.description):
        stage = ann_to_stage.get(str(desc).strip().lower())
        if stage is None:
            continue
        items.append((int(round(onset / EPOCH_SEC)),
                      max(1, int(round(dur / EPOCH_SEC))),
                      stage_y[stage]))

    levels = np.full(max(s + n for s, n, _ in items), np.nan)
    for s, n, level in items:
        levels[s:s + n] = level
    return levels


def best_offset(levels, sub, candidates=range(-3, 4)):
    best = (0, -1.0)
    for off in candidates:
        same = total = 0
        for r in sub.itertuples():
            for e in range(int(r.epoch_start), int(r.epoch_end) + 1):
                if 0 <= e + off < len(levels):
                    total += 1
                    same += int(levels[e + off] == stage_y[r.actual])
        agree = same / total if total else 0.0
        if agree > best[1]:
            best = (off, agree)
    return best


def shift(levels, off):
    if off > 0:
        return levels[off:]
    if off < 0:
        return np.concatenate([np.full(-off, np.nan), levels])
    return levels


def fill_gaps(values):
    filled = values.copy()
    valid = np.where(~np.isnan(filled))[0]
    if len(valid) == 0:
        return filled
    filled[:valid[0]] = filled[valid[0]]
    for i in range(valid[0] + 1, len(filled)):
        if np.isnan(filled[i]):
            filled[i] = filled[i - 1]
    return filled


def build_three_lines(subject, sub):
    full_gt = load_full_night(subject)
    off, agree = best_offset(full_gt, sub)
    full_gt = shift(full_gt, off)
    n_sel_seq = len(sub)
    n_sel_ep = int((sub["epoch_end"] - sub["epoch_start"] + 1).sum())
    print(f"{subject}: epochs = {n_sel_ep} | sequence = {n_sel_seq}")

    n_ep = max(len(full_gt), int(sub["epoch_end"].max()) + 1)
    full_gt = np.concatenate([full_gt, np.full(n_ep - len(full_gt), np.nan)])

    sel_gt = np.full(n_ep, np.nan)
    pred = np.full(n_ep, np.nan)
    ok = np.full(n_ep, -1, dtype=int)

    for r in sub.itertuples():
        sl = slice(int(r.epoch_start), int(r.epoch_end) + 1)
        sel_gt[sl] = stage_y[r.actual]
        pred[sl] = stage_y[r.predicted]
        ok[sl] = int(r.correct)
    return full_gt, sel_gt, pred, ok


for subject in test_table["subject"].unique():
    sub = test_table[test_table["subject"] == subject].sort_values("epoch_start").copy()
    full_gt, sel_gt, pred, ok = build_three_lines(subject, sub)
    n_ep = len(full_gt)

    x_start = np.arange(n_ep) * EPOCH_SEC / 3600
    x_end = (np.arange(n_ep) + 1) * EPOCH_SEC / 3600
    x_step = np.append(x_start, x_end[-1])
    full_step = np.append(full_gt, full_gt[-1])

    cx = (sub["epoch_start"].to_numpy() + sub["epoch_end"].to_numpy() + 1) / 2 * EPOCH_SEC / 3600
    sx0 = sub["epoch_start"].to_numpy() * EPOCH_SEC / 3600
    sx1 = (sub["epoch_end"].to_numpy() + 1) * EPOCH_SEC / 3600
    gy = sub["actual"].map(stage_y).to_numpy()
    py = sub["predicted"].map(stage_y).to_numpy()
    correct = sub["correct"].to_numpy()
    ok_color = np.where(correct, "green", "red")

    fig, axes = plt.subplots(3, 1, figsize=(FIG_W, FIG_H), sharex=True)

    axes[0].step(x_step, full_step, where="post", color="black", linewidth=LINE_W)

    axes[1].step(x_step, full_step, where="post", color="#9e9e9e", linewidth=LINE_W, zorder=1)
    axes[1].hlines(gy, sx0, sx1, color=GT_COLOR, linewidth=BLOCK_W, zorder=2)
    axes[1].scatter(cx, gy, s=MARKER_SIZE, color=GT_COLOR,
                    edgecolor="white", linewidth=1, zorder=3)

    axes[2].step(x_step, full_step, where="post", color="#9e9e9e", linewidth=LINE_W, zorder=1)
    axes[2].hlines(py, sx0, sx1, color=ok_color, linewidth=BLOCK_W, zorder=2)
    axes[2].scatter(cx, py, s=MARKER_SIZE, color=ok_color,
                    edgecolor="white", linewidth=1, zorder=3)

    for ax in axes[1:]:
        for x in cx:
            ax.axvline(x, color=GT_COLOR, linestyle="--", linewidth=0.8, alpha=0.4, zorder=0)

    axes[0].set_title(f"Real all night | {subject}")
    axes[1].set_title(f"Ground Truth (sequences used) | {subject}")
    axes[2].set_title(
        f"Predicted | {subject} | Correct {int(correct.sum())}/{len(sub)} "
        f"({correct.mean():.1%})"
    )

    for ax in axes:
        ax.set_ylabel("Sleep stage")
        ax.set_yticks(list(stage_y.values()))
        ax.set_yticklabels(list(stage_y.keys()))
        ax.set_ylim(-0.5, 4.5)
        ax.xaxis.set_major_locator(MultipleLocator(0.5))
        ax.grid(True, alpha=0.3)

    axes[0].set_xlim(-0.05, x_end[-1] * 1.12)

    axes[1].legend(handles=[
        Line2D([0], [0], color="#9e9e9e", lw=LINE_W, label="Real all night"),
        Line2D([0], [0], color=GT_COLOR, lw=4, label="Ground Truth (selected sequence)"),
    ], loc="lower right", framealpha=0.95)
    axes[2].set_xlabel("Time since recording start (hours)")
    axes[2].legend(handles=[
        Line2D([0], [0], color="#9e9e9e", lw=LINE_W, label="Real all night"),
        Line2D([0], [0], color="green", lw=4, label="Predicted: Correct"),
        Line2D([0], [0], color="red", lw=4, label="Predicted: Incorrect"),
    ], loc="lower right", framealpha=0.95)

    plt.tight_layout()
    plt.savefig(hyp_dir / f"hypnogram_full_{subject}.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()

## 9.0 ค่าตั้งต้นร่วมของกราฟสัญญาณ

In [ ]:
PLOT_CHANNELS = SELECTED_CHANNELS
BIN = 1

stage_color = {"W": "#ffd6a5", "N1": "#fdffb6", "N2": "#a0c4ff", "N3": "#bdb2ff", "R": "#ffc6ff"}

test_subject_list = [
    "SN010", "SN013", "SN020", "SN021", "SN024", "SN028", "SN031", "SN038",
    "SN047", "SN058", "SN067", "SN079", "SN084", "SN085", "SN087", "SN101",
    "SN114", "SN117", "SN136", "SN137", "SN141", "SN146", "SN147",
]
available = set(test_table["subject"])
plot_subjects = [s for s in test_subject_list if s in available]
print(f"วาด {len(plot_subjects)} subjects จาก Test set (BIN={BIN})")

X_all = DATA["test"]["X"]
ch_idx = [SELECTED_CHANNELS.index(c) for c in PLOT_CHANNELS]
n_sig = len(ch_idx)
n_bin = X_all.shape[1] // BIN


def channel_type(name):
    if name == "Chin":
        return "EMG"
    if name[0] == "E" and name[1].isdigit():
        return "EOG"
    return "EEG"


def select_sequences(subject):
    sub = test_table[test_table["subject"] == subject].sort_values("epoch_start")

    keep, last_end = [], -1
    for i, r in sub.iterrows():
        if r["epoch_start"] > last_end:
            keep.append(i)
            last_end = r["epoch_end"]
    return sub.loc[keep]


## 9.1 สัญญาณดิบทั้ง subject เทียบระยะจริงกับที่โมเดลทำนาย (Test, แยกรายคน)

In [ ]:
signal_dir = RESULT_DIR / "signal_whole_recording"
signal_dir.mkdir(parents=True, exist_ok=True)

for subject in plot_subjects:
    sub = select_sequences(subject)

    xs, los, his = [], [], []
    for i, r in sub.iterrows():
        t0 = r["epoch_start"] * 30 / 3600
        seg = np.asarray(X_all[i][: n_bin * BIN, :])[:, ch_idx].reshape(n_bin, BIN, n_sig)
        xs.append(np.append(t0 + (np.arange(n_bin) + 0.5) * BIN / FS / 3600, np.nan))
        los.append(np.vstack([seg.min(axis=1), np.full((1, n_sig), np.nan)]))
        his.append(np.vstack([seg.max(axis=1), np.full((1, n_sig), np.nan)]))
    x = np.concatenate(xs)
    lo = np.concatenate(los)
    hi = np.concatenate(his)

    fig, axes = plt.subplots(
        n_sig + 3, 1, figsize=(20, 1.2 * n_sig + 3.5), sharex=True,
        gridspec_kw={"height_ratios": [2] * n_sig + [1, 1, 0.6]}
    )

    for k in range(n_sig):
        axes[k].plot(x, hi[:, k], color="blue", linewidth=0.3)
        lim = np.nanpercentile(np.abs(np.concatenate([lo[:, k], hi[:, k]])), 99.5)
        axes[k].set_ylim(-lim, lim)
        axes[k].set_ylabel(f"{PLOT_CHANNELS[k]}\n[{channel_type(PLOT_CHANNELS[k])}] microvolt", fontsize=8)
        axes[k].grid(alpha=0.3)

    for i, r in sub.iterrows():
        t0 = r["epoch_start"] * 30 / 3600
        t1 = (r["epoch_end"] + 1) * 30 / 3600
        axes[-3].axvspan(t0, t1, facecolor=stage_color[r["actual"]], linewidth=0)
        axes[-2].axvspan(t0, t1, facecolor=stage_color[r["predicted"]], linewidth=0)
        axes[-1].axvspan(t0, t1, facecolor="green" if r["correct"] else "red", linewidth=0)
        for k in range(n_sig):
            axes[k].axvspan(t0, t1, facecolor="green" if r["correct"] else "red",
                            alpha=0.1, linewidth=0, zorder=0)

    for ax, name in [(axes[-3], "Ground Truth"), (axes[-2], "Predicted"), (axes[-1], "Correct?")]:
        ax.set_yticks([])
        ax.set_ylabel(name, rotation=0, ha="right", va="center")

    axes[-1].set_xlim(sub["epoch_start"].min() * 30 / 3600, (sub["epoch_end"].max() + 1) * 30 / 3600)
    axes[-1].set_xlabel("Time since recording start (hours)")

    axes[0].legend(handles=[
        Line2D([0], [0], color=stage_color[s], lw=8, label=s) for s in STAGES
    ] + [
        Line2D([0], [0], color="green", lw=8, label="Correct"),
        Line2D([0], [0], color="red", lw=8, label="Wrong"),
    ], loc="lower right", bbox_to_anchor=(1, 1.01), ncol=7, fontsize=8)

    n_ok = int(sub["correct"].sum())
    fig.suptitle(
        f"{subject} | whole recording | correct {n_ok}/{len(sub)} sequences ({n_ok / len(sub):.1%}) "
        f"(blank = no sequence)"
    )
    plt.tight_layout()
    plt.savefig(signal_dir / f"signal_whole_{subject}.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()

## 9.2 สัญญาณของแต่ละ sequence เทียบระยะการนอนจริงกับที่โมเดลทำนาย (Test, แยกรายคน)

In [ ]:
FIG_W = 30
GAP = 0.25
seq_sec = X_all.shape[1] / FS

signal_dir = RESULT_DIR / "signal_sequences"
signal_dir.mkdir(parents=True, exist_ok=True)

for subject in plot_subjects:
    sub = select_sequences(subject)
    rows = list(sub.iterrows())
    n_seq = len(rows)

    xs, los, his = [], [], []
    for k, (i, r) in enumerate(rows):
        x0 = k * (1 + GAP)
        seg = np.asarray(X_all[i][: n_bin * BIN, :])[:, ch_idx].reshape(n_bin, BIN, n_sig)
        rel = (np.arange(n_bin) + 0.5) * BIN / FS / seq_sec
        xs.append(np.append(x0 + rel, np.nan))
        los.append(np.vstack([seg.min(axis=1), np.full((1, n_sig), np.nan)]))
        his.append(np.vstack([seg.max(axis=1), np.full((1, n_sig), np.nan)]))
    x = np.concatenate(xs)
    lo = np.concatenate(los)
    hi = np.concatenate(his)

    fig, axes = plt.subplots(
        n_sig + 3, 1, figsize=(FIG_W, 1.2 * n_sig + 3.5), sharex=True,
        gridspec_kw={"height_ratios": [2] * n_sig + [1, 1, 0.6]}
    )

    for k in range(n_sig):
        axes[k].plot(x, hi[:, k], color="blue", linewidth=0.3)
        lim = np.nanpercentile(np.abs(np.concatenate([lo[:, k], hi[:, k]])), 99.5)
        axes[k].set_ylim(-lim, lim)
        axes[k].set_ylabel(f"{PLOT_CHANNELS[k]}\n[{channel_type(PLOT_CHANNELS[k])}] microvolt", fontsize=8)
        axes[k].grid(alpha=0.3)

    centers, labels = [], []
    for k, (i, r) in enumerate(rows):
        t0, t1 = k * (1 + GAP), k * (1 + GAP) + 1
        axes[-3].axvspan(t0, t1, facecolor=stage_color[r["actual"]], linewidth=0)
        axes[-2].axvspan(t0, t1, facecolor=stage_color[r["predicted"]], linewidth=0)
        axes[-1].axvspan(t0, t1, facecolor="green" if r["correct"] else "red", linewidth=0)
        for kk in range(n_sig):
            axes[kk].axvspan(t0, t1, facecolor="green" if r["correct"] else "red",
                             alpha=0.1, linewidth=0, zorder=0)

        axes[-3].text((t0 + t1) / 2, 0.5, r["actual"], ha="center", va="center",
                      fontsize=10, fontweight="bold", transform=axes[-3].get_xaxis_transform())
        axes[-2].text((t0 + t1) / 2, 0.5, r["predicted"], ha="center", va="center",
                      fontsize=10, fontweight="bold", transform=axes[-2].get_xaxis_transform())
        centers.append((t0 + t1) / 2)
        labels.append(f"{r['epoch_start'] * 30 / 3600:.2f} h\n(epoch {int(r['epoch_start'])}-{int(r['epoch_end'])})")

    for k in range(n_seq - 1):
        xg = k * (1 + GAP) + 1 + GAP / 2
        for ax in axes:
            ax.axvline(xg, color="gray", linestyle=":", linewidth=1)

    for ax, name in [(axes[-3], "Ground Truth"), (axes[-2], "Predicted"), (axes[-1], "Correct?")]:
        ax.set_yticks([])
        ax.set_ylabel(name, rotation=0, ha="right", va="center")

    axes[-1].set_xlim(-GAP / 2, n_seq * (1 + GAP) - GAP / 2)
    axes[-1].set_xticks(centers)
    axes[-1].set_xticklabels(labels, fontsize=9)
    axes[-1].set_xlabel("Selected sequences in time order (gaps removed; tick = real time since recording start)")

    axes[0].legend(handles=[
        Line2D([0], [0], color=stage_color[s], lw=8, label=s) for s in STAGES
    ] + [
        Line2D([0], [0], color="green", lw=8, label="Correct"),
        Line2D([0], [0], color="red", lw=8, label="Wrong"),
    ], loc="lower right", bbox_to_anchor=(1, 1.01), ncol=7, fontsize=8)

    n_ok = int(sub["correct"].sum())
    fig.suptitle(
        f"{subject} | selected sequences | correct {n_ok}/{len(sub)} sequences ({n_ok / len(sub):.1%})"
    )
    plt.tight_layout()
    plt.savefig(signal_dir / f"signal_sequences_{subject}.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()

## 10. ตารางภาพรวมทุก subject: ทายถูก/ผิดในแต่ละระยะจริง (Test)

In [ ]:
pivot = test_table.pivot_table(index="subject", columns="actual", values="predicted", aggfunc="first")[STAGES]
ok = pivot == np.array(STAGES)

order = ok.sum(axis=1).sort_values(ascending=False, kind="stable").index
pivot, ok = pivot.loc[order], ok.loc[order]
n = len(ok)

green = [0.78, 0.92, 0.78]
red = [0.96, 0.70, 0.70]
colors = np.where(ok.to_numpy()[:, :, None], green, red)

fig, ax = plt.subplots(figsize=(8, 0.4 * n + 2))
ax.imshow(colors, aspect="auto")

for i in range(n):
    for j in range(5):
        text = "OK" if ok.iloc[i, j] else "Pred " + pivot.iloc[i, j]
        ax.text(j, i, text, ha="center", va="center", fontsize=9)

ax.set_xticks(range(5))
ax.set_xticklabels([f"Actual {s}\n{ok[s].sum()}/{n}" for s in STAGES])
ax.xaxis.tick_top()
ax.set_yticks(range(n))
ax.set_yticklabels([f"{s} ({ok.loc[s].sum()}/5)" for s in ok.index], fontsize=8)
ax.set_title("Test: green = correct, red = wrong (Pred X = model prediction)", pad=40)
plt.tight_layout()
plt.savefig(RESULT_DIR / "proof_table_test.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. รายละเอียดรายคน: ระยะจริง, สัญญาณ 30 วินาที, ความน่าจะเป็นของโมเดล (Test)

In [ ]:
def visualize_subject(split, subject):
    meta = DATA[split]["meta"]
    idx = meta.index[meta["subject"] == subject].to_numpy()
    idx = idx[np.argsort(DATA[split]["y"][idx], kind="stable")]
    table = tables[split]
    t = np.arange(EPOCH_LEN) / FS

    fig, axes = plt.subplots(N_CHANNELS + 1, len(idx), figsize=(3.3 * len(idx), 2 * (N_CHANNELS + 1)), squeeze=False)

    for col, i in enumerate(idx):
        row = table.loc[i]
        signal = DATA[split]["X"][i][-EPOCH_LEN:, :]
        color = "green" if row["correct"] else "red"

        axes[0, col].set_title(f"Expert: {row['actual']}\nModel: {row['predicted']}", color=color, fontweight="bold")

        for c in range(N_CHANNELS):
            axes[c, col].plot(t, signal[:, c], lw=0.6)
            axes[c, col].set_xlim(0, 30)
            axes[c, col].grid(alpha=0.3)

        probs = [row["prob_" + s] for s in STAGES]
        axes[-1, col].bar(STAGES, probs, color="lightgray")
        axes[-1, col].bar(row["predicted"], row["prob_" + row["predicted"]], color=color)
        axes[-1, col].set_ylim(0, 1)
        axes[-1, col].set_xlabel("Probability")

    for c in range(N_CHANNELS):
        low = min(axes[c, col].get_ylim()[0] for col in range(len(idx)))
        high = max(axes[c, col].get_ylim()[1] for col in range(len(idx)))
        for col in range(len(idx)):
            axes[c, col].set_ylim(low, high)
        axes[c, 0].set_ylabel(SELECTED_CHANNELS[c])

    plt.suptitle(f"{split} | {subject} | raw signal (microvolt) of the last 30 s epoch")
    plt.tight_layout()

    out_dir = RESULT_DIR / "visualize"
    out_dir.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_dir / f"{split}_{subject}.png", dpi=150, bbox_inches="tight")
    plt.show()


per_subject = test_table.groupby("subject")["correct"].sum().sort_values()
picks = np.linspace(0, len(per_subject) - 1, min(5, len(per_subject)), dtype=int)
selected_subjects = per_subject.index[picks].tolist()
display(per_subject.loc[selected_subjects])

for subject in selected_subjects:
    visualize_subject("test", subject)